In [ ]:
!pip install sentence-transformers textblob

In [ ]:
import random
import time
import re
from textblob import TextBlob
from sentence_transformers import SentenceTransformer, util

In [ ]:
def detect_emotion(text):
    sentiment = TextBlob(text).sentiment.polarity

    if sentiment > 0.3:
        return "positive"
    elif sentiment < -0.3:
        return "negative"
    else:
        return "neutral"

In [ ]:
intents = {
    "greeting": [
        "hi", "hello", "good morning", "hey there"
    ],
    "order_issue": [
        "where is my order", "order not delivered", "track my parcel"
    ],
    "refund_issue": [
        "refund", "return my money", "cancel my order"
    ],
    "technical_problem": [
        "app crashed", "error message", "not working"
    ]
}

In [ ]:
responses = {
    "greeting": {
        "neutral": "Hi there! How can I help you today?",
        "positive": "Hey! 😊 Great to see you. What can I do for you?",
        "negative": "Hello. I hope everything is okay. Tell me how I can assist."
    },

    "order_issue": {
        "neutral": "Let me quickly check that for you. Could you share your order ID?",
        "negative": "I'm really sorry for the inconvenience. Let me track it immediately. Can you share your order ID?",
        "positive": "Sure! Let’s track that order together. Please share your order ID."
    },

    "refund_issue": {
        "neutral": "Refunds are processed within 5–7 working days.",
        "negative": "I understand how frustrating that can be. I’ll help you with the refund process right away.",
        "positive": "No worries! I’ll guide you through the refund process."
    },

    "technical_problem": {
        "neutral": "Can you describe the issue in more detail?",
        "negative": "I’m really sorry you’re facing this issue. Let’s fix it together. What error are you seeing?",
        "positive": "Got it! Let’s solve this quickly. What exactly is happening?"
    },

    "fallback": {
        "neutral": "I want to make sure I give you accurate information. Let me connect you with a human support specialist.",
        "negative": "I don't want to give you incorrect guidance. I'll immediately escalate this to a human agent for priority support.",
        "positive": "Let me double-check this with our support team to ensure you get the best solution."
    }
}

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def detect_intent(user_input):
    user_embedding = model.encode(user_input, convert_to_tensor=True)

    best_intent = None
    highest_score = 0

    for intent, examples in intents.items():
        example_embeddings = model.encode(examples, convert_to_tensor=True)
        similarity = util.cos_sim(user_embedding, example_embeddings)
        score = float(similarity.max())

        if score > highest_score:
            highest_score = score
            best_intent = intent

    return best_intent, highest_score

In [ ]:
def detect_intent(user_input):
    user_embedding = model.encode(user_input, convert_to_tensor=True)

    best_intent = None
    highest_score = 0

    for intent, examples in intents.items():
        example_embeddings = model.encode(examples, convert_to_tensor=True)
        similarity = util.cos_sim(user_embedding, example_embeddings)
        score = float(similarity.max())

        if score > highest_score:
            highest_score = score
            best_intent = intent

    return best_intent, highest_score

In [ ]:
conversation_history = []

def generate_response(user_input):
    emotion = detect_emotion(user_input)
    intent, confidence = detect_intent(user_input)

    conversation_history.append(user_input)

    if confidence < 0.45:
        intent = "fallback"

    reply = responses[intent][emotion]

    return reply, confidence, emotion

In [ ]:
def human_typing(text):
    for char in text:
        print(char, end="", flush=True)
        time.sleep(0.02)
    print()

In [ ]:
print("Empathetic Cognitive Support Agent Activated.")
print("Type 'exit' to end conversation.\n")

while True:
    user_input = input("You: ")

    if user_input.lower() == "exit":
        print("Session ended.")
        break

    reply, confidence, emotion = generate_response(user_input)

    print("Bot: ", end="")
    human_typing(reply)

    print(f"(Emotion Detected: {emotion} | Confidence: {round(confidence,2)})")
    print("-"*50)

Empathetic Cognitive Support Agent Activated.
Type 'exit' to end conversation.

You: exit
Session ended.
